# P03 — Prédiction de la consommation énergétique d'un logement (DPE)

**Contexte métier.** Diagnostiqueurs, agences de rénovation, ADEME, banques (éco-PTZ) : estimer la consommation d'un logement avant audit complet, cibler les passoires thermiques. Enjeu : 15,8 millions de DPE en base, objectif national de rénovation.

**Objectif de l'EDA.** Identifier la cible (consommation kWh/m²/an) et les régresseurs (période de construction, isolation, zone climatique, surface).

**Données.** DPE logements existants (depuis juillet 2021) — ADEME via l'API data-fair de data.ademe.fr. Licence ouverte Etalab 2.0. Base complète : 15 807 887 lignes ; l'EDA utilise un échantillon de 8 000 lignes via l'API `/lines`.

## 1. Téléchargement (échantillon via l'API data-fair)

L'API ADEME expose la base DPE en JSON paginé ; on récupère 8 000 logements avec les champs utiles seulement (`select=`), ce qui évite de télécharger la base de 15,8 M de lignes.

In [ ]:
import urllib.request, urllib.parse, json, os
os.makedirs("data", exist_ok=True)

CHAMPS = ",".join([
    "etiquette_dpe", "etiquette_ges", "type_batiment", "annee_construction",
    "periode_construction", "surface_habitable_logement", "hauteur_sous_plafond",
    "zone_climatique", "code_departement_ban", "conso_5_usages_par_m2_ep",
    "qualite_isolation_murs", "qualite_isolation_menuiseries",
    "type_installation_chauffage", "emission_ges_5_usages_par_m2"])
URL = ("https://data.ademe.fr/data-fair/api/v1/datasets/"
       "meg-83tjwtg8dyz4vv7h1dqe/lines?size=8000&select="
       + urllib.parse.quote(CHAMPS, safe=","))
if not os.path.exists("data/dpe_sample.json"):
    urllib.request.urlretrieve(URL, "data/dpe_sample.json")
d = json.load(open("data/dpe_sample.json"))
print("Total en base :", d["total"], "— échantillon :", len(d["results"]))

## 2. Chargement

L'échantillon JSON est converti en DataFrame pandas ; la consommation par m² est la cible continue.

In [ ]:
import json
import pandas as pd, numpy as np

d = json.load(open("data/dpe_sample.json"))
df = pd.DataFrame(d["results"])
df["conso"] = pd.to_numeric(df["conso_5_usages_par_m2_ep"], errors="coerce")
df = df.dropna(subset=["conso"])
print(len(df), "logements avec conso renseignée")
print(df["etiquette_dpe"].value_counts().to_dict())

## 3. Analyse

Consommation médiane par étiquette DPE, par période de construction, par qualité d'isolation et par zone climatique — cohérence de la cible avec l'étiquette et corrélations.

In [ ]:
print("Conso médiane (kWh/m²/an) par étiquette DPE :")
print(df.groupby("etiquette_dpe")["conso"].median().round(0).to_dict())
print()
print("Par période de construction :")
print(df.groupby("periode_construction")["conso"].median().round(0).to_dict())
print()
print("Par isolation des murs :", df.groupby("qualite_isolation_murs")["conso"].median().round(0).to_dict())
print("Par zone climatique :", df.groupby("zone_climatique")["conso"].median().round(0).to_dict())
num = df[["conso", "surface_habitable_logement", "hauteur_sous_plafond"]].apply(pd.to_numeric, errors="coerce")
print("Corrélations Spearman :", num.corr("spearman")["conso"].round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- total base dpe : 15807887
- n echantillon : 8000
- etiquette dpe dist : C : 2935, D : 2458, E : 1271, B : 510, F : 442, G : 239, A : 145
- type batiment dist : appartement : 5120, maison : 2622, immeuble : 258
- conso m2 stats : count : 8 000.00, mean : 199.60, std : 101.80, min : 19.70, 25% : 127.00, 50% : 182.00, 75% : 248.00, max : 1 447.00
- conso mediane par etiquette : A : 60.00, B : 92.00, C : 132.00, D : 205.00, E : 283.00, F : 373.00, G : 490.00
- conso mediane par periode construction : 1948-1974 : 199.00, 1975-1977 : 194.00, 1978-1982 : 184.00, 1983-1988 : 176.00, 1989-2000 : 167.00, 2001-2005 : 142.0…
- conso mediane par isolation murs : bonne : 151.00, insuffisante : 225.00, moyenne : 167.00, très bonne : 101.00
- conso mediane par isolation menuiseries : bonne : 171.00, insuffisante : 242.00, moyenne : 200.00, très bonne : 128.00
- conso mediane par zone climatique : H1a : 196.00, H1b : 195.00, H1c : 201.00, H2a : 170.00, H2b : 172.00, H2c : 167.00, H2d : 163.00, H3 : 140.00
- corr spearman conso : conso_5_usages_par_m2_ep : 1.00, surface_habitable_logement : -0.23, hauteur_sous_plafond : 0.19

## 5. Variable cible et régresseurs

**Cible (régression) : `conso_5_usages_par_m2_ep` (kWh ep/m²/an).** Cible continue directement liée à l'étiquette DPE ; une classification A-G est un objectif secondaire possible.

**Régresseurs retenus :**
- `periode_construction` : avant 1948 → 247 kWh/m² médian contre 86 pour 2013-2021 — le signal le plus fort
- `qualite_isolation_murs` (225 médiane si insuffisante vs 101 si très bonne), `qualite_isolation_menuiseries`
- `zone_climatique` (H1 plus froid que H3) et `code_departement_ban`
- `type_batiment` (maison vs appartement), `surface_habitable_logement`, `hauteur_sous_plafond`
- `type_installation_chauffage`

**Pièges :**
- `code_departement_ban` manquant à 10,7 % : imputation ou exclusion à décider
- L'étiquette DPE (`etiquette_dpe`) est directement calculée depuis la cible : fuite de données, à exclure des features en régression
- Échantillon de 8 000 lignes suffisant pour l'EDA, mais pour l'entraînement, paginer l'API (`after=`) ou télécharger la base complète
- Les DPE sont déclaratifs/audités selon deux méthodologies : ne pas mélanger sans contrôle